# Band structure and dielectric function of the parabolic two-band model

Analytic and numerical ingredients of the parabolic two-band model (adapted from the lecture notes of Shunsuke Sato):

1. the parabolic valence and conduction bands,
2. the imaginary part of the dielectric function in **2D** (numerical k integral with Lorentzian broadening, checked against the analytic step),
3. the same quantity in **3D** (square-root absorption edge).

No simulation data is needed; everything is computed from the model parameters. All calculations use atomic units ($\hbar = e = m_e = 1$) and are converted to eV for plotting.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

import twoband as tb

: 

## Model parameters

In [ ]:
M_V = 0.5  # hole effective mass
M_C = 0.3  # electron effective mass
E_GAP = 4.45 / tb.HARTREE_TO_EV  # band gap (Hartree)
P_VC = 0.96  # interband momentum matrix element
M_R = tb.reduced_mass(M_C, M_V)

print(f"reduced mass m_r = {M_R:.4f}")

## 1. Parabolic bands

$$\epsilon_{v}(k) = -\frac{k^{2}}{2 m_{v}}, \qquad \epsilon_{c}(k) = \epsilon_{g} + \frac{k^{2}}{2 m_{c}}$$

In [ ]:
k = np.linspace(-0.5, 0.5, 1001)  # a.u.

plt.plot(k, tb.conduction_band(k, M_C, E_GAP) * tb.HARTREE_TO_EV, label="Conduction band", lw=2)
plt.plot(k, tb.valence_band(k, M_V) * tb.HARTREE_TO_EV, label="Valence band", lw=2)
plt.xlabel("k (a.u.)")
plt.ylabel("Energy (eV)")
plt.title("Parabolic band structure")
plt.legend()
plt.show()

## 2. Im[$\epsilon(\omega)$] in 2D

$$\mathrm{Im}\,\epsilon(\omega) = \left(\frac{2\pi}{\omega}\right)^{2} \frac{2}{(2\pi)^{2}} \int d^{2}k\, |p_{vc}|^{2}\, \delta\!\left(\epsilon_{c}(k) - \epsilon_{v}(k) - \omega\right)$$

The delta function is replaced by a Lorentzian of width $\eta$. For parabolic bands the joint density of states in 2D is constant, so the exact result is a step at the gap:

$$\mathrm{Im}\,\epsilon(\omega) = \left(\frac{2\pi}{\omega}\right)^{2} \frac{2}{(2\pi)^{2}}\, |p_{vc}|^{2}\, 2\pi m_r \quad \text{for } \omega > \epsilon_g$$

The rise at the lowest energies in the numerical curve is the tail of the Lorentzian broadening, amplified by the $1/\omega^2$ prefactor. It is not physical and shrinks as $\eta \to 0$.

In [ ]:
omega = np.linspace(0.5, 20.0, 500) / tb.HARTREE_TO_EV
eta = 0.05 / tb.HARTREE_TO_EV

eps_2d = tb.im_eps_2d(omega, E_GAP, M_R, P_VC, eta=eta)
eps_2d_exact = np.where(
    omega > E_GAP,
    (2 * np.pi / omega) ** 2 * 2 / (2 * np.pi) ** 2 * P_VC**2 * 2 * np.pi * M_R,
    0.0,
)

plt.plot(
    omega * tb.HARTREE_TO_EV,
    eps_2d,
    lw=2,
    label=rf"numerical, $\eta$ = {eta * tb.HARTREE_TO_EV:.2f} eV",
)
plt.plot(omega * tb.HARTREE_TO_EV, eps_2d_exact, "k--", label="analytic")
plt.xlabel("Energy (eV)")
plt.ylabel(r"Im[$\epsilon(\omega)$]")
plt.title("2D parabolic two-band model")
plt.legend()
plt.show()

## 3. Im[$\epsilon(\omega)$] in 3D

In 3D the joint density of states grows as a square root above the gap:

$$\mathrm{Im}\,\epsilon(\omega) = 2\,(2 m_r)^{3/2}\,|p_{vc}|^{2}\,\frac{\sqrt{\omega - \epsilon_g}}{\omega^{2}} \quad \text{for } \omega > \epsilon_g$$

Here with GaAs-like parameters ($m_r = 0.067$).

In [ ]:
M_R_3D = 0.067
P_VC_3D = 0.9

freq = np.linspace(0.01, 10.0, 2000) / tb.HARTREE_TO_EV
eps_3d = tb.im_eps_3d(freq, E_GAP, M_R_3D, P_VC_3D)

plt.plot(freq * tb.HARTREE_TO_EV, eps_3d, "r", lw=2)
plt.xlabel(r"$\omega$ (eV)")
plt.ylabel(r"Im[$\epsilon(\omega)$] (a.u.)")
plt.title("3D parabolic two-band model")
plt.xlim(0, 10)
plt.show()

## Reference

S. A. Sato et al., *Applied Sciences* 8(10), 1777 (2018). https://www.mdpi.com/2076-3417/8/10/1777